# P-027 — Bootstrap Confidence Interval on Tau-b

**Decision:** What is the 95% confidence interval on our headline tau-b, and can we honestly claim 0.27?

| Item | Value |
|------|-------|
| Dataset | perovskite_stability_clean.csv (1,543 devices) |
| Features | 16, fillna(0) |
| Model | Locked ExtraTreesRegressor (n_estimators=700, max_features='sqrt', min_samples_split=3, min_samples_leaf=1, bootstrap=False, random_state=42) |
| Current claim | tau-b ≈ 0.271 (seed-42 split), CV ≈ 0.289 (5-fold) |
| Method | 1,000 bootstrap resamples → train/test → tau-b each → percentile CI |

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import kendalltau
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split

# Load data
df_raw = pd.read_csv("perovskite_stability_clean.csv")
print(f"Raw dataset: {df_raw.shape[0]} devices, {df_raw.shape[1]} columns")

# Locked 16 features (from P-024 audit)
FEATURES = [
    "Perovskite_band_gap",
    "Pb", "Sn", "I", "Br", "Cl",
    "MA", "FA", "Cs",
    "first_Prvskt_annealing_temperature",
    "first_Prvskt_thermal_annealing_time",
    "Perovskite_thickness",
    "Cell_area_measured",
    "JV_default_Voc", "JV_default_Jsc", "JV_default_FF",
]

target = "Stability_PCE_T80"
X = df_raw[FEATURES].fillna(0)
y = np.log1p(df_raw[target].fillna(0))

print(f"Features: {X.shape[1]}, target: log1p({target})")
print(f"Devices: {len(X)}")
print(f"y range: [{y.min():.3f}, {y.max():.3f}]")

Raw dataset: 1543 devices, 47 columns
Features: 16, target: log1p(Stability_PCE_T80)
Devices: 1543
y range: [0.002, 9.036]


In [2]:
import time

N_BOOT = 1000
rng = np.random.RandomState(2024)
tau_values = []
top20_overlaps = []  # for Cell 5 panel metric

et_params = dict(
    n_estimators=700,
    max_features='sqrt',
    min_samples_split=3,
    min_samples_leaf=1,
    bootstrap=False,
    random_state=42,
    n_jobs=-1,
)

t0 = time.time()
for i in range(N_BOOT):
    # Resample WITH replacement (same size as original)
    idx = rng.choice(len(X), size=len(X), replace=True)
    X_boot = X.iloc[idx].reset_index(drop=True)
    y_boot = y.iloc[idx].reset_index(drop=True)

    # 80/20 split with seed=42
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_boot, y_boot, test_size=0.2, random_state=42
    )

    # Train locked ET and predict
    model = ExtraTreesRegressor(**et_params)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)

    # Kendall tau-b
    tau, _ = kendalltau(y_te, preds)
    tau_values.append(tau)

    # Top-20 overlap (pred vs true in test set)
    n_te = len(y_te)
    if n_te >= 20:
        true_top20 = set(np.argsort(np.array(y_te))[-20:])
        pred_top20 = set(np.argsort(preds)[-20:])
        top20_overlaps.append(len(true_top20 & pred_top20))
    else:
        top20_overlaps.append(np.nan)

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        print(f"  Iteration {i+1:4d}/{N_BOOT}  |  tau-b = {tau:.4f}  |  elapsed {elapsed:.0f}s")

tau_arr = np.array(tau_values)
overlap_arr = np.array(top20_overlaps)
print(f"\nDone. {N_BOOT} bootstraps in {time.time()-t0:.0f}s")

  Iteration  100/1000  |  tau-b = 0.6445  |  elapsed 25s


  Iteration  200/1000  |  tau-b = 0.5693  |  elapsed 49s


  Iteration  300/1000  |  tau-b = 0.5581  |  elapsed 74s


  Iteration  400/1000  |  tau-b = 0.5562  |  elapsed 98s


  Iteration  500/1000  |  tau-b = 0.5720  |  elapsed 123s


  Iteration  600/1000  |  tau-b = 0.5925  |  elapsed 148s


  Iteration  700/1000  |  tau-b = 0.5869  |  elapsed 173s


  Iteration  800/1000  |  tau-b = 0.5856  |  elapsed 198s


  Iteration  900/1000  |  tau-b = 0.5501  |  elapsed 223s


  Iteration 1000/1000  |  tau-b = 0.6380  |  elapsed 249s

Done. 1000 bootstraps in 249s


In [3]:
# --- Bootstrap statistics ---
mean_tau = np.mean(tau_arr)
se_tau = np.std(tau_arr, ddof=1)
ci95_lo, ci95_hi = np.percentile(tau_arr, [2.5, 97.5])
ci99_lo, ci99_hi = np.percentile(tau_arr, [0.5, 99.5])

print("=" * 55)
print("  Bootstrap results  (1,000 resamples)")
print("=" * 55)
print(f"  Mean tau-b       : {mean_tau:.4f}")
print(f"  Std error        : {se_tau:.4f}")
print(f"  95% CI           : [{ci95_lo:.4f}, {ci95_hi:.4f}]")
print(f"  99% CI           : [{ci99_lo:.4f}, {ci99_hi:.4f}]")
print(f"  Min / Max        : {tau_arr.min():.4f} / {tau_arr.max():.4f}")
print()

# Key checks
zero_inside = ci95_lo <= 0.0 <= ci95_hi
print(f"  0.0  inside 95% CI?  {'YES — no evidence of power' if zero_inside else 'NO — strong evidence of predictive power'}")

pt20_inside = ci95_lo <= 0.20 <= ci95_hi
print(f"  0.20 inside 95% CI?  {'YES — cannot rule out tau < 0.20' if pt20_inside else 'NO — entire CI above 0.20, claim justified'}")
print("=" * 55)

  Bootstrap results  (1,000 resamples)
  Mean tau-b       : 0.5977
  Std error        : 0.0353
  95% CI           : [0.5250, 0.6618]
  99% CI           : [0.5023, 0.6886]
  Min / Max        : 0.4618 / 0.7087

  0.0  inside 95% CI?  NO — strong evidence of predictive power
  0.20 inside 95% CI?  NO — entire CI above 0.20, claim justified


In [4]:
# --- Panel-relevant metric: top-20 overlap across bootstraps ---
# How many of the true top-20 test-set devices does the model place in its predicted top-20?

valid_overlaps = overlap_arr[~np.isnan(overlap_arr)]
n_test = int(len(X) * 0.2)
random_baseline = 20 * 20 / n_test

print("=" * 55)
print("  Top-20 Overlap (predicted vs true) across bootstraps")
print("=" * 55)
print(f"  Mean overlap     : {np.mean(valid_overlaps):.1f} / 20 devices")
print(f"  Median overlap   : {np.median(valid_overlaps):.0f} / 20 devices")
print(f"  Std              : {np.std(valid_overlaps):.2f}")
print(f"  Min / Max        : {np.min(valid_overlaps):.0f} / {np.max(valid_overlaps):.0f}")
print(f"  Random baseline  : {random_baseline:.1f} / 20 (test size = {n_test})")
print(f"  Lift over random : {np.mean(valid_overlaps) / random_baseline:.1f}x")
print(f"  Bootstraps with overlap >= 5 : {(valid_overlaps >= 5).sum()} / {len(valid_overlaps)}")
print(f"  Bootstraps with overlap >= 10: {(valid_overlaps >= 10).sum()} / {len(valid_overlaps)}")
print("=" * 55)

  Top-20 Overlap (predicted vs true) across bootstraps
  Mean overlap     : 11.9 / 20 devices
  Median overlap   : 12 / 20 devices
  Std              : 2.21
  Min / Max        : 4 / 18
  Random baseline  : 1.3 / 20 (test size = 308)
  Lift over random : 9.2x
  Bootstraps with overlap >= 5 : 999 / 1000
  Bootstraps with overlap >= 10: 859 / 1000


In [5]:
# --- Save bootstrap tau-b values ---
out_df = pd.DataFrame({"bootstrap_iter": range(1, N_BOOT + 1), "tau_b": tau_arr})
out_path = "Packet_P027_Bootstrap_CI.csv"
out_df.to_csv(out_path, index=False)
print(f"Saved {out_path}  ({len(out_df)} rows)")
print(out_df.describe())

Saved Packet_P027_Bootstrap_CI.csv  (1000 rows)
       bootstrap_iter        tau_b
count     1000.000000  1000.000000
mean       500.500000     0.597656
std        288.819436     0.035314
min          1.000000     0.461842
25%        250.750000     0.574992
50%        500.500000     0.596900
75%        750.250000     0.619620
max       1000.000000     0.708672


In [6]:
# --- Status footer ---
print("=" * 55)
print("  P-027 STATUS FOOTER")
print("=" * 55)

if ci95_lo > 0.15:
    status = "CONFIRMED"
    verdict = f"95% CI [{ci95_lo:.3f}, {ci95_hi:.3f}] is entirely above 0.15 — strong evidence of predictive power."
elif ci95_lo > 0.10:
    status = "PROMISING"
    verdict = f"95% CI [{ci95_lo:.3f}, {ci95_hi:.3f}] is above 0.10 but not above 0.15."
else:
    status = "NEGATIVE"
    verdict = f"95% CI [{ci95_lo:.3f}, {ci95_hi:.3f}] includes values near or below 0 — weak evidence."

print(f"  Status  : {status}")
print(f"  Verdict : {verdict}")
print(f"  Claim 'tau-b ~ 0.27' : {'Supported' if ci95_lo <= 0.271 <= ci95_hi else 'Point estimate outside CI'}")
print(f"  Mean bootstrap tau-b : {mean_tau:.4f}")
print(f"  SE                   : {se_tau:.4f}")
print("=" * 55)

  P-027 STATUS FOOTER
  Status  : CONFIRMED
  Verdict : 95% CI [0.525, 0.662] is entirely above 0.15 — strong evidence of predictive power.
  Claim 'tau-b ~ 0.27' : Point estimate outside CI
  Mean bootstrap tau-b : 0.5977
  SE                   : 0.0353
